In [1]:
import random
import csv
import osmnx as ox

In [2]:
# REAL STREETS IN HEERLEN with their postcode ranges and valid house numbers
# Sources: Postcode data from various Heerlen streets [citation:2][citation:6][citation:7]
HEERLEN_STREETS = [
    # Street name, base postcode, min house number, max house number
    ("Kerkstraat", "6411", 1, 50),        # Central Heerlen [citation:1]
    ("Stationsstraat", "6411", 1, 40),     # Near station area [citation:2]
    ("Dorpsstraat", "6412", 1, 60),        # [citation:2]
    ("Schoolstraat", "6413", 1, 45),       # [citation:2]
    ("Akerstraat", "6411", 1, 120),        # Major street [citation:2]
    ("Bongerd", "6411", 1, 30),            # City center
    ("Gasthuisstraat", "6411", 1, 25),     # Near hospital
    ("Pancratiusstraat", "6411", 1, 20),   # Church area
    ("Trompstraat", "6412", 1, 35),        # Residential
    ("Valkenburgerweg", "6412", 1, 80),    # Main road
    ("Heerlerbaan", "6418", 1, 277),       # [citation:7] - extensive range
    ("Heerlerheide", "6413", 1, 100),      # District
    ("Putgraaf", "6411", 1, 45),           # City center [citation:2]
    ("Raadhuisstraat", "6411", 1, 30),     # Town hall area
    ("Lindelaan", "6414", 1, 50),          # Residential
    ("Beukenlaan", "6414", 1, 45),         # Residential
    ("Eikenlaan", "6414", 1, 40),          # Residential
    ("Wilhelminastraat", "6412", 1, 55),   # [citation:2]
    ("Julianaweg", "6413", 1, 60),         # Residential
    ("Prinses Irenestraat", "6413", 1, 35), # Residential
    ("Tollensstraat", "6416", 1, 42),      # [citation:6][citation:8] - exact numbers
    ("Jacob van Maerlantstraat", "6416", 2, 34),  # [citation:9] - even numbers only
    ("Eisterweg", "6422", 2, 8),            # [citation:10]
    ("Hondsdraf", "6418", 1, 15),           # [citation:1]
    ("Lienaertsstraat", "6416", 1, 45),     # [citation:1] - corrected to Heerlen
    ("Welterlaan", "6415", 1, 45),          # [citation:3]
    ("Laan van Hövell tot Westerflier", "6411", 1, 44),  # [citation:4]
    ("Caumerweg", "6418", 1, 94),           # [citation:7]
    ("Pastoor Erensstraat", "6418", 1, 34), # [citation:7]
    ("September 1944-straat", "6418", 1, 121), # [citation:7]
    ("Palestinastraat", "6418", 1, 279),    # [citation:7] - long street
    ("Nazarethstraat", "6418", 1, 96),      # [citation:7]
    ("Jeruzalemstraat", "6418", 2, 46),     # [citation:7] - even numbers
    ("Bautscherweg", "6418", 1, 166),       # [citation:7]
    ("Corisbergweg", "6418", 20, 205),      # [citation:7]
    ("A gen Giezen", "6418", 1, 64),        # [citation:7]
]

# Street names without full data - will use with generic postcode
ADDITIONAL_STREETS = [
    ("Ds. Jongeneelstraat", "6411", 1, 30),
    ("Kapelaan Berixstraat", "6411", 1, 25),
    ("Op de Nobel", "6411", 1, 20),
    ("Bekkerweg", "6411", 1, 35),
    ("Deken Nicolaijestraat", "6411", 1, 28),
    ("Kortstraat", "6411", 1, 15),
    ("Oude Lindestraat", "6411", 1, 22),
    ("Ambachtsstraat", "6411", 1, 18),
    ("Tempsplein", "6411", 1, 10),
    ("Coriovallumstraat", "6411", 1, 40),
    ("Ruys de Beerenbroucklaan", "6411", 1, 60),
    ("Mariabad", "6411", 1, 12),
    ("Oliemolenstraat", "6411", 1, 30),
    ("Keerweg", "6418", 28, 115),           # [citation:7]
    ("Vrijheidstraat", "6418", 1, 19),       # [citation:7]
    ("Montgomerystraat", "6418", 1, 18),     # [citation:7]
    ("Vredestraat", "6418", 1, 20),          # [citation:7]
    ("Pattonstraat", "6418", 1, 49),         # [citation:7]
    ("Hodgesstraat", "6418", 1, 50),         # [citation:7]
    ("Herlongstraat", "6418", 1, 59),        # [citation:7]
    ("Horicherhofstraat", "6418", 1, 32),    # [citation:7]
    ("Caumerboord", "6418", 1, 99),          # [citation:7]
    ("Hambeukerboord", "6418", 1, 101),      # [citation:7]
    ("Bradleystraat", "6418", 1, 31),        # [citation:7]
    ("Oud Valkenhuizerstraat", "6418", 4, 4), # [citation:7] - single number
    ("Wienweg", "6418", 7, 92),              # [citation:7]
    ("Bovenste Caumer", "6418", 2, 24),      # [citation:7]
    ("Simpsonstraat", "6418", 2, 10),        # [citation:7]
    ("Bergdriesch", "6418", 1, 65),          # [citation:7]
    ("Kanaalstraat", "6418", 1, 37),         # [citation:7]
    ("Jerichostraat", "6418", 1, 98),        # [citation:7]
    ("Giezenhof", "6418", 1, 19),            # [citation:7]
    ("Sinaïstraat", "6418", 2, 28),          # [citation:7]
    ("Samariastraat", "6418", 1, 35),        # [citation:7]
    ("Judeastraat", "6418", 1, 39),          # [citation:7]
    ("Bethlehemstraat", "6418", 1, 58),      # [citation:7]
    ("Galileastraat", "6418", 1, 48),        # [citation:7]
    ("Romeinenstraat", "6418", 2, 32),        # [citation:7]
]

In [3]:
# Combine all streets
ALL_STREETS = HEERLEN_STREETS + ADDITIONAL_STREETS

# Possible care arrangements
CARE_ARRANGEMENTS = ["HBH Basic", "HBH Plus", "Wash & Ironing", "V&V"]
CARE_HOURS = [1, 1.5, 2, 2.5, 3]   # step 0.5

# Enable OSMnx cache so repeated validation requests stay fast.
ox.settings.use_cache = True

# Cache address validation results to avoid repeated geocoder calls.
_GEOCODE_EXISTS_CACHE = {}

def _address_exists_in_heerlen(street: str, house_number: int) -> bool:
    query = f"{street} {house_number}, Heerlen, Netherlands"
    if query in _GEOCODE_EXISTS_CACHE:
        return _GEOCODE_EXISTS_CACHE[query]

    try:
        ox.geocode(query)
        _GEOCODE_EXISTS_CACHE[query] = True
    except Exception:
        _GEOCODE_EXISTS_CACHE[query] = False

    return _GEOCODE_EXISTS_CACHE[query]

def generate_real_address(max_attempts: int = 200):
    """Generate a geocode-validated Heerlen address from the configured street list."""
    for _ in range(max_attempts):
        street, _base_postcode, min_num, max_num = random.choice(ALL_STREETS)

        # Generate house number within configured range.
        house_number = random.randint(min_num, max_num)

        # Respect known parity constraint for this street.
        if street == "Jacob van Maerlantstraat" and house_number % 2 != 0:
            house_number = house_number + 1 if house_number < max_num else house_number - 1

        if _address_exists_in_heerlen(street, house_number):
            return f"{street} {house_number}, Heerlen"

    raise RuntimeError(
        "Could not generate a valid real address in Heerlen after "
        f"{max_attempts} attempts. Check street ranges and connectivity."
    )

def generate_client(index):
    """Generate a single client record as a dictionary."""
    name = f"Client {index}"
    address = generate_real_address()
    care_arrangement = random.choice(CARE_ARRANGEMENTS)
    preference = random.choice(["morning", "afternoon"])

    if preference == "morning":
        time_window_start = "08:00"
        time_window_end   = "12:00"
    else:
        time_window_start = "12:00"
        time_window_end   = "18:00"

    care_hours = random.choice(CARE_HOURS)

    # Most clients have no pets; occasional 1 or 2
    dogs = random.choices([0, 1, 2], weights=[0.7, 0.2, 0.1])[0]
    cats = random.choices([0, 1, 2], weights=[0.6, 0.3, 0.1])[0]

    # 30% chance of smoking
    smokes = random.random() < 0.3

    return {
        "name": name,
        "address": address,
        "care_arrangement": care_arrangement,
        "preferences": preference,
        "time_window_start": time_window_start,
        "time_window_end": time_window_end,
        "care_hours": care_hours,
        "dogs": dogs,
        "cats": cats,
        "smokes": smokes
    }

def generate_clients_csv(num_clients, filename="../output/clients.csv"):
    """Generate num_clients records and write them to a CSV file."""
    from pathlib import Path

    fieldnames = [
        "name", "address", "care_arrangement", "preferences",
        "time_window_start", "time_window_end", "care_hours",
        "dogs", "cats", "smokes"
    ]

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, mode='w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for i in range(1, num_clients + 1):
            client = generate_client(i)
            # Convert boolean to lowercase string for readability
            client["smokes"] = str(client["smokes"]).lower()
            writer.writerow(client)

    print(f"Generated {num_clients} clients in '{output_path}'.")

if __name__ == "__main__":
    # Generate 100 clients by default
    generate_clients_csv(100)

Generated 100 clients in '..\output\clients.csv'.
